# Oceananigans validation of the flat-bottom PG Green's-function solution

This notebook compares a weak, horizontally uniform, rotating Boussinesq column computed with Oceananigans against the analytical Green's-function solution. The flat bottom and periodic horizontal directions isolate the shared vertical physics: no-slip bottom velocity, surface stress, prescribed surface buoyancy, and interior relaxation toward `B(z)`.

Oceananigans solves the full nonhydrostatic Boussinesq equations, not the planetary-geostrophic reduction. The comparison is therefore an asymptotic weak-flow test. It deliberately uses no horizontal buoyancy gradient or surface-pressure gradient; a closed-basin/topographic comparison is a separate validation problem.

Oceananigans emits an upstream support warning on Julia 1.13 because it is tested through Julia 1.12. This CPU notebook is smoke-tested here on Julia 1.13; retain the warning with benchmark results.

In [ ]:
using Oceananigans
using LinearAlgebra
using Printf
using Statistics

include(normpath(joinpath(pwd(), "..", "TheoryNotes", "VelocityFieldPrototype.jl")))
using .VelocityFieldPrototype


## Matched weak-flow case

The coefficients below use the same nondimensional convention as `PG_Solution_v0.1.ipynb`: Oceananigans viscosity and diffusivity are `Ek * nu` and `Ek * kappa`, respectively. Oceananigans surface momentum flux is kinematic and uses the opposite sign to the notebook's positive `∂z u = τx` convention, so it is `-nu_ocean * tau_analytical`.

In [ ]:
const Lx = 1.0
const Ly = 1.0
const H = 1.0
const f0 = 1.0
const Ek = 0.02
const Γ = 0.2
const nu_kernel = 1.0
const kappa_kernel = 1.0
const nu_ocean = Ek * nu_kernel
const kappa_ocean = Ek * kappa_kernel

# Keep the Rossby number small: the analytical solution has no inertia.
const tau_analytical = 1e-4 + 0.0im
const b_surface = 2e-3
const B_gradient = 1e-2

Hfun(x, y) = H
ffun(x, y) = f0
zero2(x, y) = 0.0
Bfun(x, y, z) = B_gradient * z
bsfun(x, y) = b_surface

parameters = VelocityParameters(Ek, Γ, nu_kernel, kappa_kernel, f0, 0.0)
kernels = load_symbolic_kernels()

b_reference(z) = buoyancy_from_forcing(parameters, Hfun, Bfun, bsfun, 0.0, 0.0, z)
u_reference(z) = velocity_b23_from_gradps(
    kernels, parameters, Hfun, ffun, zero2, zero2, zero2,
    (x, y) -> tau_analytical, Bfun, bsfun, zero2, zero2, 0.0, 0.0, z,
)

# Oceananigans calls continuous boundary functions with x, y, z.
surface_buoyancy(x, y, z) = b_surface
target_buoyancy(x, y, z, t) = B_gradient * z

@show nu_ocean kappa_ocean Γ tau_analytical b_surface


In [ ]:
function make_simulation(Nz; Nx=8, Ny=8, Δt=0.01, stop_time=20.0)
    grid = RectilinearGrid(CPU();
        topology = (Periodic, Periodic, Bounded),
        size = (Nx, Ny, Nz),
        x = (0, Lx), y = (0, Ly), z = (-H, 0),
    )

    u_bcs = FieldBoundaryConditions(
        top = FluxBoundaryCondition(-nu_ocean * real(tau_analytical)),
        bottom = ValueBoundaryCondition(0.0),
    )
    v_bcs = FieldBoundaryConditions(
        top = FluxBoundaryCondition(-nu_ocean * imag(tau_analytical)),
        bottom = ValueBoundaryCondition(0.0),
    )
    b_bcs = FieldBoundaryConditions(
        top = ValueBoundaryCondition(surface_buoyancy),
        bottom = GradientBoundaryCondition(0.0),
    )

    model = NonhydrostaticModel(grid;
        advection = nothing,
        buoyancy = BuoyancyTracer(),
        coriolis = FPlane(f=f0),
        tracers = :b,
        closure = ScalarDiffusivity(ν=nu_ocean, κ=kappa_ocean),
        boundary_conditions = (u=u_bcs, v=v_bcs, b=b_bcs),
        forcing = (b=Relaxation(rate=Γ, target=target_buoyancy),),
    )

    # Starting from the analytical buoyancy profile isolates momentum spin-up.
    set!(model, u=0, v=0, w=0, b=(x, y, z) -> b_reference(z))
    return Simulation(model; Δt, stop_time)
end

function relative_l2(numerical, reference)
    return norm(numerical .- reference) / max(norm(reference), eps(eltype(numerical)))
end

function compare_with_green_function(simulation)
    model = simulation.model
    u = Array(interior(model.velocities.u))
    v = Array(interior(model.velocities.v))
    b = Array(interior(model.tracers.b))
    z = collect(znodes(model.tracers.b))

    u_exact = reshape(real.(u_reference.(z)), 1, 1, :)
    v_exact = reshape(imag.(u_reference.(z)), 1, 1, :)
    b_exact = reshape(b_reference.(z), 1, 1, :)

    metrics = (
        u_relative_l2 = relative_l2(u, u_exact),
        v_relative_l2 = relative_l2(v, v_exact),
        b_relative_l2 = relative_l2(b, b_exact),
        max_speed = maximum(sqrt.(u.^2 .+ v.^2)),
    )
    @show metrics
    return metrics
end


## Accuracy and performance tests

The warm-up run compiles Oceananigans and is intentionally excluded from the timing. Increase `spinup_time` if velocity errors have not plateaued. Compare the error trend over `Nz` rather than relying on a single tolerance, because Oceananigans uses a staggered finite-volume discretization while the reference is continuous.

In [ ]:
const spinup_time = 100.0

# JIT warm-up; discard this timing and result.
warmup = make_simulation(8; stop_time=0.1)
run!(warmup)

function run_accuracy_case(Nz; Nx=8, Ny=8, stop_time=spinup_time)
    simulation = make_simulation(Nz; Nx, Ny, stop_time)
    run!(simulation)
    return (; Nz, metrics=compare_with_green_function(simulation))
end

accuracy_results = [run_accuracy_case(Nz) for Nz in (8, 16, 32)]
display(accuracy_results)

function warm_run_seconds(Nz; Nx=8, Ny=8, stop_time=spinup_time)
    GC.gc()
    start = time_ns()
    simulation = make_simulation(Nz; Nx, Ny, stop_time)
    run!(simulation)
    return (time_ns() - start) * 1e-9
end

performance_results = [
    (; Nz, seconds=median(warm_run_seconds(Nz) for _ in 1:3))
    for Nz in (8, 16, 32)
]
display(performance_results)
